<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### TaskReview


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [8]:
using System;
using System.Collections.Generic;
using System.Linq;


public enum OrderStatus
{
    Created,   
    InWork,    
    Completed  
}


public delegate void OrderStatusChangedHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);


public interface ICompany
{
    void AddEmployee(Employee employee);
    void AddOrder(Order order);
    void AssignEmployeeToOrder(int orderId, Employee employee);
    void ChangeOrderStatus(int orderId, OrderStatus newStatus);
    List<Order> GetOrdersByStatus(OrderStatus status);
    List<Order> GetOrdersByCustomer(string firstName, string lastName);
    void PrintEmployeesReport();
    
    void AddCustomer(Customer customer);
    List<Customer> GetCustomers();
}

public interface IPerson
{
    string FirstName { get; }
    string LastName { get; }
    string GetFullName();
}


public abstract class Person : IPerson
{
    public string FirstName { get; }
    public string LastName { get; }

    protected Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public virtual string GetFullName() => $"{FirstName} {LastName}";
}


public class Employee : Person
{
    public string Position { get; }
    public int OrdersProcessed { get; private set; }

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public void IncrementOrdersProcessed() => OrdersProcessed++;

    public override string GetFullName() => $"{FirstName} {LastName} ({Position})";

    public void NotifyOrderStatusChanged(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        Console.WriteLine($"  [Уведомление] {GetFullName()}: заказ #{order.OrderId} " +
                          $"изменил статус с «{Order.TranslateStatus(oldStatus)}» на «{Order.TranslateStatus(newStatus)}».");
    }
}


public class Customer : Person
{
    
    public string PhoneNumber { get; private set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public override string GetFullName() => $"{FirstName} {LastName} (тел. {PhoneNumber})";

    
    public void UpdatePhoneNumber(string newPhoneNumber)
    {
        if (!string.IsNullOrWhiteSpace(newPhoneNumber))
            PhoneNumber = newPhoneNumber;
    }

    public string GetContactInfo() => $"{FirstName} {LastName}: {PhoneNumber}";

    public override string ToString() => GetContactInfo();
}


public class Order
{
    private static int _nextId = 1;

    public int OrderId { get; }
    public string Description { get; }
    public Employee AssignedEmployee { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; }
    public Customer Customer { get; }

    public event OrderStatusChangedHandler StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = _nextId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        AssignedEmployee = employee;
        Console.WriteLine($"  Заказу #{OrderId} назначен сотрудник: {employee.GetFullName()}");
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus) return;

        var oldStatus = Status;
        Status = newStatus;

        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
            AssignedEmployee.IncrementOrdersProcessed();

        StatusChanged?.Invoke(this, oldStatus, newStatus);
    }

    public override string ToString()
    {
        var employee = AssignedEmployee != null ? AssignedEmployee.GetFullName() : "не назначен";
        return $"#{OrderId} | {Description} | Заказчик: {Customer.GetFullName()} | " +
               $"Сотрудник: {employee} | Статус: {TranslateStatus(Status)} | Создан: {CreatedAt:dd.MM.yyyy HH:mm}";
    }

    public static string TranslateStatus(OrderStatus status) => status switch
    {
        OrderStatus.Created => "Создан",
        OrderStatus.InWork => "В работе",
        OrderStatus.Completed => "Завершен",
        _ => status.ToString()
    };
}


public class Company : ICompany
{
    public string Name { get; }
    private readonly List<Employee> _employees = new List<Employee>();
    private readonly List<Order> _orders = new List<Order>();
    // [FIX] отдельный список заказчиков, как требует ТЗ
    private readonly List<Customer> _customers = new List<Customer>();

    public Company(string name) => Name = name;

    public void AddEmployee(Employee employee)
    {
        if (employee != null && !_employees.Contains(employee))
            _employees.Add(employee);
    }

    
    public void AddCustomer(Customer customer)
    {
        if (customer != null && !_customers.Contains(customer))
            _customers.Add(customer);
    }

    public List<Customer> GetCustomers() => new List<Customer>(_customers);

    public void AddOrder(Order order)
    {
        if (order == null) return;
        _orders.Add(order);
        // [FIX] автоматически регистрируем заказчика, если его ещё нет
        if (order.Customer != null && !_customers.Contains(order.Customer))
            _customers.Add(order.Customer);
        order.StatusChanged += OnOrderStatusChanged;
    }

    private void OnOrderStatusChanged(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        order.AssignedEmployee?.NotifyOrderStatusChanged(order, oldStatus, newStatus);
    }

    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        var order = _orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine($"  Заказ #{orderId} не найден.");
            return;
        }
        order.AssignEmployee(employee);
    }

    public void ChangeOrderStatus(int orderId, OrderStatus newStatus)
    {
        var order = _orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine($"  Заказ #{orderId} не найден.");
            return;
        }
        order.ChangeStatus(newStatus);
    }

    public List<Order> GetOrdersByStatus(OrderStatus status)
        => _orders.Where(o => o.Status == status).ToList();

    public List<Order> GetOrdersByCustomer(string firstName, string lastName)
        => _orders.Where(o => o.Customer.FirstName == firstName && o.Customer.LastName == lastName).ToList();

    public List<Employee> GetEmployees() => new List<Employee>(_employees);
    public List<Order> GetOrders() => new List<Order>(_orders);

    public void PrintEmployeesReport()
    {
        Console.WriteLine($"Отчёт по сотрудникам компании «{Name}»:");
        if (_employees.Count == 0)
        {
            Console.WriteLine("  Сотрудников нет.");
            return;
        }
        foreach (var e in _employees)
            Console.WriteLine($"  {e.GetFullName()} — обработано заказов: {e.OrdersProcessed}");
    }
}


var company = new Company("ООО «Ромашка»");

var ivan = new Employee("Иван", "Иванов", "Менеджер");
var anna = new Employee("Анна", "Петрова", "Разработчик");
company.AddEmployee(ivan);
company.AddEmployee(anna);

var customer1 = new Customer("Пётр", "Сидоров", "+7-900-111-22-33");
var customer2 = new Customer("Мария", "Кузнецова", "+7-900-444-55-66");

company.AddCustomer(customer1);
company.AddCustomer(customer2);

var order1 = new Order("Разработка сайта", customer1);
var order2 = new Order("Настройка сети", customer2);
var order3 = new Order("Консультация по CRM", customer1);
company.AddOrder(order1);
company.AddOrder(order2);
company.AddOrder(order3);

Console.WriteLine("=== Все заказы ===");
foreach (var o in company.GetOrders())
    Console.WriteLine(o);

Console.WriteLine("\n=== Назначение сотрудников ===");
company.AssignEmployeeToOrder(order1.OrderId, anna);
company.AssignEmployeeToOrder(order2.OrderId, ivan);
company.AssignEmployeeToOrder(order3.OrderId, anna);

Console.WriteLine("\n=== Изменение статусов (с уведомлениями) ===");
company.ChangeOrderStatus(order1.OrderId, OrderStatus.InWork);
company.ChangeOrderStatus(order1.OrderId, OrderStatus.Completed);
company.ChangeOrderStatus(order2.OrderId, OrderStatus.Completed);
company.ChangeOrderStatus(order3.OrderId, OrderStatus.InWork);

Console.WriteLine("\n=== Заказы в работе ===");
foreach (var o in company.GetOrdersByStatus(OrderStatus.InWork))
    Console.WriteLine(o);

Console.WriteLine("\n=== Завершённые заказы ===");
foreach (var o in company.GetOrdersByStatus(OrderStatus.Completed))
    Console.WriteLine(o);

Console.WriteLine("\n=== Поиск заказов по заказчику: Пётр Сидоров ===");
foreach (var o in company.GetOrdersByCustomer("Пётр", "Сидоров"))
    Console.WriteLine(o);


Console.WriteLine("\n=== Заказчики компании ===");
foreach (var c in company.GetCustomers())
    Console.WriteLine($"  {c.GetContactInfo()}");

Console.WriteLine();
company.PrintEmployeesReport();

=== Все заказы ===
#1 | Разработка сайта | Заказчик: Пётр Сидоров (тел. +7-900-111-22-33) | Сотрудник: не назначен | Статус: Создан | Создан: 20.11.2026 22:53
#2 | Настройка сети | Заказчик: Мария Кузнецова (тел. +7-900-444-55-66) | Сотрудник: не назначен | Статус: Создан | Создан: 20.11.2026 22:53
#3 | Консультация по CRM | Заказчик: Пётр Сидоров (тел. +7-900-111-22-33) | Сотрудник: не назначен | Статус: Создан | Создан: 20.11.2026 22:53

=== Назначение сотрудников ===
  Заказу #1 назначен сотрудник: Анна Петрова (Разработчик)
  Заказу #2 назначен сотрудник: Иван Иванов (Менеджер)
  Заказу #3 назначен сотрудник: Анна Петрова (Разработчик)

=== Изменение статусов (с уведомлениями) ===
  [Уведомление] Анна Петрова (Разработчик): заказ #1 изменил статус с «Создан» на «В работе».
  [Уведомление] Анна Петрова (Разработчик): заказ #1 изменил статус с «В работе» на «Завершен».
  [Уведомление] Иван Иванов (Менеджер): заказ #2 изменил статус с «Создан» на «Завершен».
  [Уведомление] Анна Петро